# Ch.4  時系列データの異常を検知する

受講生用ノートブック

| 項目 | 内容 |
|------|------|
| 使うデータ | 人工的な時系列データ（工場センサーのイメージ） |
| 演習時間 | 35分（問3まで完了で合格） |
| ゴール | LinearRegression を使って「過去の傾向からのズレ」を数値化し、異常点を検出する |

---
## Copilot の使い方

URL: https://copilot.microsoft.com

Copilot への伝え方のコツ（5点セット）：
```
【知りたいこと】〇〇をしたい / 〇〇を計算したい
【使うライブラリ】pandas / matplotlib / scikit-learn
【データの形】df という DataFrame、120行×3列（time / value / lag_1）
【環境】Python 3.8.6、JupyterLab
【困っていること】どう書けばよいかわからない
```

💡 Ch.4 の一言アドバイス：
「shift() や predict() など、聞いたことのないメソッドは
 『df の value 列を1行ずらして lag_1 列に入れたい、pandas でどう書くか』のように
 列名・操作・結果をセットで伝えましょう。」

時系列の操作は慣れないと難しいですが、**やりたいことを日本語で具体的に説明する**だけで
Copilot が書いてくれます。

---
## 準備  ライブラリとデータを生成する

⛔ 注意 このセルを実行するとデータが生成されます。必ず最初に実行してください。

In [ ]:
import numpy as np                               # 数値計算・乱数生成
import pandas as pd                              # データ操作の基本ライブラリ
import matplotlib.pyplot as plt                  # グラフ描画
import japanize_matplotlib                       # 日本語フォント対応
from sklearn.linear_model import LinearRegression # 線形回帰モデル

# 再現性のため乱数シードを固定する（同じ結果が毎回出るようにする）
np.random.seed(42)
n    = 120                                       # データ件数

# 正常な時系列データを生成する（緩やかな上昇トレンド＋小さなノイズ）
time  = np.arange(n)                             # 0, 1, 2, ..., 119
base  = 50 + 0.3 * time + np.random.normal(0, 2, n)
# 上の式: 50 → 初期値、0.3*time → 上昇トレンド、random.normal → ランダムノイズ

# 異常期間（80〜89 番目）にスパイク（急激な外れ値）を追加する
base[80:90] += np.random.uniform(20, 35, 10)    # 20〜35 の値を足して異常値を作る

df = pd.DataFrame({"time": time, "value": base})

---
## なぜ「予測との差」で異常を検知するのか

異常検知の考え方（LinearRegression を使う方法）：

1. 正常期間のデータでモデルを学習させる（「正常とはこういうもの」を学ぶ）
2. 全期間でモデルが「予測する値」を計算する
3. 「実際の値」と「予測値」のズレ（残差）を計算する
4. ズレが大きい点 = 正常パターンから外れた = 「異常の疑い」

実務での活用：
- 工場機器の温度・振動センサーで「故障の前兆」を検知
- 売上データで「急激な増減（キャンペーン効果 or 問題）」を検知

---
## STEP 1  データを「目で」確認する

どこにスパイク（急激な変動）があるかをグラフで見て、感覚を掴みましょう。
機械に任せる前に、「人間の目でも明らかな異常があるか」を確認することが大切です。

---

Q1-1：折れ線グラフで時系列データを表示してください

time を X 軸、value を Y 軸にして、データ全体の推移を確認してください。

💡 ヒント：「2列を使って折れ線グラフを描く方法」を Copilot に聞いてみましょう。

In [ ]:
# time を X 軸、value を Y 軸にした折れ線グラフで全体の推移を確認する
plt.figure(figsize=(12, 4))
plt.plot(df["time"], df["value"])       # 折れ線グラフを描く
plt.title("センサー値の時系列推移")
plt.xlabel("time（時刻）")
plt.ylabel("value（センサー値）")
plt.show()

---
STEP 1（追加）  グラフの前にデータの基本統計を確認しましょう

センサーデータを受け取ったとき、まず「平均・最大・最小・標準偏差」を確認します。
スパイク（異常）が入っていると、平均や std が「正常期間だけ」と比べてどう変わるか確認しましょう。

💡 ヒント：「DataFrame の基本統計量を確認する方法」を Copilot に聞いてみましょう。

In [ ]:
# データ全体の基本統計量を確認して、スパイクが統計値に与える影響を把握する
print(df.describe().to_string())

---
### STEP 1 の気づき（AI 禁止：1 行でOK）

Q：スパイク（急激な変動）は何番目付近に見えましたか？また、正常期間はどのような形をしていましたか？

>

---
## STEP 2  「過去の値」を新しい列として追加する（ラグ特徴量）

なぜラグ特徴量が必要か：
LinearRegression に「時刻 t の値」を予測させるには、「時刻 t-1 の値」を入力として使います。
「前の時点の値」を新しい列として追加することを「ラグ特徴量を作る」と言います。

---

Q2-1：1 時点前の値（lag_1）を追加してください

`shift()` を使って、1 行前の value 列の値を新しい列として追加してください。

💡 ヒント：「列の値を1行分ずらした新しい列を作る方法」を Copilot に聞いてみましょう。

---
### shift() を使うと何が起きるか

`shift(1)` を実行すると、先頭の行が **NaN（空欄）** になります。

```
実行前:          実行後:
time  value     time  value  lag_1
  0   50.6        0   50.6   NaN   ← 先頭行は「1つ前」がないのでNaN
  1   51.3        1   51.3   50.6
  2   49.8        2   49.8   51.3
```

NaN のまま計算できないので `dropna()` で先頭行を削除します。
**1行減るのは正常な動作です。**

In [ ]:
# 1時点前の値をラグ特徴量（lag_1）として新しい列に追加する

# shift(1) → 各行の値を1行下にずらす（= 1時点前の値になる）
df["lag_1"] = df["value"].shift(1)

# shift(1) の結果、先頭行の lag_1 が NaN になるため削除する
df = df.dropna().reset_index(drop=True)   # dropna → NaN行を削除、reset_index → 行番号を0から振り直す

---
STEP 2（追加）  lag_1 が正しく作れているか先頭を確認しましょう

`shift()` を実行したあと、`df.head()` で `lag_1` 列が正しく追加されているか目で確認します。
「time=1 の lag_1」が「time=0 の value」と同じ値になっていれば成功です。

💡 ヒント：「DataFrame の先頭を表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# 先頭数行で time・value・lag_1 の対応関係を確認する
df.head()

---
### STEP 2 の気づき（AI 禁止：1 行でOK）

Q：`df.head()` を確認して、lag_1 列に何が入っているか説明してください。先頭行が NaN だった場合はどうしましたか？

>

---
## 問1  正常期間だけでモデルを学習させる ★☆☆

実務でのイメージ：
「故障前の正常データでモデルを作り、そのモデルで予測した値と現在値のズレを異常スコアとして使う」

---

Q：最初の 80 点（正常期間）だけで LinearRegression のモデルを学習させてください

- 正常期間：time が 0〜79（インデックスで最初の 80 行）
- X（入力）：lag_1 列
- y（答え）：value 列

💡 ヒント：「特定の行だけ取り出して LinearRegression を学習させる方法」を Copilot に聞いてみましょう。

In [ ]:
# 正常期間（time < 80）のデータだけを抽出して、学習用の X と y を準備する

train   = df[df["time"] < 80]       # time が 80 未満の行だけを取り出す（正常期間）
X_train = train[["lag_1"]]          # 入力：1時点前の値（列名をリストで指定 → DataFrame 形式）
y_train = train["value"]            # 正解：現在の値

In [ ]:
# LinearRegression モデルを作成して、正常期間のデータで学習させる

model = LinearRegression()
model.fit(X_train, y_train)         # fit → 訓練データでモデルを学習させる
print("学習完了 ✅")

---
問1-a（追加）  正常期間の学習データ件数を確認しましょう

正常期間（time が 0〜79）を取り出すと何行が学習に使われるか確認します。
Ch.3 の `X_train` 件数確認と同じ習慣です。

📌 この 80行でモデルを学習させるため、time 80〜89 のスパイクはモデルが「見ていない」状態になります。

💡 ヒント：「条件に合う行だけを取り出してその行数を確認する方法」を Copilot に聞いてみましょう。

In [ ]:
# 正常期間（time < 80）のデータ件数を確認する
print(f"正常期間: {len(df[df['time'] < 80])} 件")

---
問1（追加）  学習した直線の「傾き」と「切片」を確認しましょう

座学スライドで学んだ `y = a × x + b` の a（傾き）と b（切片）が実際に何の値になったかを確認します。
このセンサーデータは「緩やかに上昇するトレンド」があるので、傾き a は正の値（1 に近い数）になるはずです。

💡 ヒント：「LinearRegression の学習後に傾きと切片を取得する方法」を Copilot に聞いてみましょう。

In [ ]:
# 学習済みモデルの「傾き（a）」と「切片（b）」を確認する

a = model.coef_[0]           # 傾き
b = model.intercept_         # 切片
print(f"傾き  a = {a:.4f}")
print(f"切片  b = {b:.4f}")
print(f"予測式: value = {a:.3f} × lag_1 + {b:.3f}")

---
### 問1 の気づき（AI 禁止：1 行でOK）

Q：正常期間（time 0〜79）だけでモデルを学習させました。なぜ異常期間のデータを学習に使わないのでしょうか？

>

---
## 問2  全期間で予測して「ズレ（残差）」を計算する ★★☆

実務でのイメージ：
「正常時の動きを学んだモデルで全データの値を予測し、実際の値との差が大きい点 = 異常候補とする」

---

Q2-1：全期間のデータでモデルが予測した値と、実際の値との差（残差）を計算してください

1. `model.predict(df[["lag_1"]])` で全 119 点の予測値を計算する
2. `df["value"] - 予測値` で残差（ズレ）を計算して `df["residual"]` に入れる
3. 残差の絶対値 `abs(残差)` を `df["score"]` に入れる

💡 ヒント：「全期間の予測値を計算して残差と絶対値を新しい列に追加する方法」を Copilot に聞いてみましょう。

In [ ]:
# 全期間の予測値を計算する（正常期間のデータで学んだパターンを使う）
df["pred"] = model.predict(df[["lag_1"]])

In [ ]:
# 残差（実際の値 - 予測値）を計算する
df["residual"] = df["value"] - df["pred"]

In [ ]:
# 残差の絶対値を「異常スコア」として計算する
df["score"] = df["residual"].abs()   # abs() → 絶対値を取る

In [ ]:
# 異常スコア（残差の絶対値）の推移を折れ線グラフで確認する
plt.figure(figsize=(12, 4))
plt.plot(df["time"], df["score"])
plt.title("異常スコア（残差の絶対値）の推移")
plt.xlabel("time（時刻）")
plt.ylabel("score（異常スコア）")
plt.show()

---
問2-3（追加）  正常期間と異常期間のスコアを比べましょう

スライドで「正常: 誤差 0〜3、異常: 誤差 30.5」と説明されました。
実際に正常期間（time < 80）と異常期間（time 80〜89）のスコア統計を比べて、
しきい値をどこに設定すべきかを数値で確認しましょう。

💡 ヒント：「条件でデータを絞り込んで describe() で統計を確認する方法」を Copilot に聞いてみましょう。

In [ ]:
# 正常期間と異常期間のスコアをそれぞれ取り出す
normal_score  = df[df["time"] < 80]["score"]                           # 正常期間
anomaly_score = df[(df["time"] >= 80) & (df["time"] < 90)]["score"]   # 異常期間

In [ ]:
# 正常期間と異常期間のスコア統計を比較する（差の大きさを数値で確認する）
print("正常期間（time 0〜79）のスコア統計:")
print(normal_score.describe().round(2))
print()
print("異常期間（time 80〜89）のスコア統計:")
print(anomaly_score.describe().round(2))

---
## 問3  しきい値で異常を判定してグラフに表示する ★★☆（ゴール）

実務でのイメージ：
「異常スコアが一定値以上の点だけをアラートとして抽出し、グラフで確認する」

---

Q3-1：正常期間（最初の 80 点）の score の 95パーセンタイルをしきい値にして、
そのしきい値を超えた点を「異常」と判定してください

1. `np.percentile(正常期間のscore, 95)` でしきい値を計算する
2. `df["score"] > しきい値` で True/False の列を作る（`df["is_anomaly"]`）

💡 ヒント：「95 パーセンタイルをしきい値にして条件に合う行に True を付ける方法」を Copilot に聞いてみましょう。

In [ ]:
# 正常期間（time < 80）のスコアを取り出す（しきい値の計算に使う）
normal_score = df[df["time"] < 80]["score"]

In [ ]:
# 95パーセンタイル値をしきい値として計算する
threshold = np.percentile(normal_score, 95)
print(f"しきい値（95パーセンタイル）: {threshold:.2f}")

In [ ]:
# しきい値を超えた点に異常フラグ（True/False）を付ける
df["is_anomaly"] = df["score"] > threshold      # しきい値を超えたら True

print(f"異常判定件数: {df['is_anomaly'].sum()} 件")

In [ ]:
# スパイク期間（time 80〜90）の検出件数を確認する
detected = df[(df["time"] >= 80) & (df["time"] < 90) & df["is_anomaly"]].shape[0]
print(f"スパイク期間の検出件数: {detected} / 10件")

---
問3-1（追加②）  異常と判定された行だけを取り出して確認しましょう

`is_anomaly=True` の行だけを抽出して、どの time の点が異常と判定されたかを一覧で確認します。
グラフの「赤い点がどこに出たか」と照合してみましょう。

💡 ヒント：「True の行だけを絞り込んで特定の列だけを表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# 異常と判定された行だけ取り出して確認する
anomaly_df = df[df["is_anomaly"]][["time", "value", "score"]].reset_index(drop=True)
print(anomaly_df.round(2))

---
### 問3-1 の気づき（AI 禁止：1 行でOK）

Q：異常判定件数は何件でしたか？スパイク期間（time 80〜90）の 10 点のうち何件が検出できたと思いますか？

>

---
Q3-2：グラフに正常点と異常点を色分けして表示してください

- 正常点：通常の折れ線グラフ
- 異常点（is_anomaly=True の点）：赤い点（散布図）で重ねて表示

💡 ヒント：「折れ線グラフに条件に合う点だけを赤い点でハイライトする方法」を Copilot に聞いてみましょう。

In [ ]:
# 時系列グラフに異常判定結果を重ね合わせて可視化する

anomalies = df[df["is_anomaly"]]           # 異常と判定された行だけ取り出す

plt.figure(figsize=(12, 4))
plt.plot(df["time"], df["value"], label="センサー値（全体）")          # 折れ線全体
plt.scatter(anomalies["time"], anomalies["value"],
            color="red", zorder=5, label="異常判定点")                # 赤い点で強調
plt.axhline(threshold + df["pred"].mean(),
            color="orange", linestyle="--", label="しきい値ライン")   # しきい値の目安ライン
plt.title("時系列異常検知の結果")
plt.xlabel("time（時刻）")
plt.ylabel("value（センサー値）")
plt.legend()
plt.show()

---
### 問3-2 の気づき（AI 禁止：1 行でOK）

Q：赤い点はどこに集まっていましたか？STEP 1 で目視したスパイクの位置と一致していましたか？

>

---
## STEP 3  振り返り （AI 禁止）

⛔ 注意 AI は使わないこと。自分の言葉で書いてください。

Q：LinearRegression を使った異常検知の仕組みを、自分の言葉で説明してください

>

Q：このモデルを実務（工場・在庫管理・売上監視など）でどう使えそうですか？

>

---
## 問4（発展）  しきい値を変えて比較する ★★★

📋 発展 時間が余った人だけ取り組んでください

---

Q：パーセンタイルを 90・95・99 で変えて、異常判定件数がどう変わるか比較してください

💡 ヒント：「パーセンタイル値を変えながら異常件数の変化を比較する方法」を Copilot に聞いてみましょう。

In [ ]:
# パーセンタイル値（90・95・99）を変えながら、しきい値と異常検出件数の変化を比較する

for pct in [90, 95, 99]:
    thr = np.percentile(normal_score, pct)           # しきい値を計算
    cnt = (df["score"] > thr).sum()                  # 異常と判定される件数
    print(f"パーセンタイル {pct}%: しきい値={thr:.2f}  →  異常検出件数={cnt} 件")

---
## お疲れさまでした！

| 操作 | 発見 | ビジネスへの接続 |
|------|------|----------------|
| 可視化 | time 80〜90 にスパイク | 目視でも確認できた |
| ラグ特徴量 | 過去の値を入力として使う | 時系列への機械学習の基本 |
| 残差計算 | 正常モデルとのズレを数値化 | 異常スコアの考え方 |
| しきい値判定 | 9 件の異常を自動検出 | 工場監視・売上モニタリングで応用可 |